# Hook Interactions

**Purpose:** Answer research question 5 from the rodin v2 proposal — how do multiple PreToolUse hooks on the same tool interact?

| # | Question | Why it matters |
|---|----------|----------------|
| 5 | How do multiple PreToolUse hooks on the same tool interact? | Rodin hooks ExitPlanMode — if other plugins also hook it, we need to know if all hooks run, if denial is short-circuiting, and what ordering applies |

Source: `openspec/changes/rodin/proposal.md` — Outstanding research question 5

Plugins: `research/plugins/hook-test-a/` and `hook-test-b/` — two independent configurable PreToolUse hooks

## Background

Claude Code supports multiple plugins, each of which can register hooks. When two plugins both register a PreToolUse hook (with or without matchers), several questions arise:

1. **Ordering**: Do hooks from plugin A always run before plugin B? Is order deterministic?
2. **Short-circuiting**: If hook A denies, does hook B still run?
3. **Aggregation**: If both allow, is the result a simple AND? If both deny, which reason is surfaced?
4. **Independence**: Do hooks see each other's responses?

### Test design

Two plugins (`hook-test-a` and `hook-test-b`) both hook PreToolUse on all tools. Each:
- Reads its response from a config file (`/tmp/claude/hook-{a,b}-response.json`)
- Logs its invocation with a timestamp to `/tmp/claude/hook-{a,b}-log.jsonl`

We configure different response combinations and observe behavior:

| Test | Hook A response | Hook B response | Expected |
|------|----------------|----------------|----------|
| 1 | `continue: true` | `continue: true` | Tool runs |
| 2 | `deny` | `continue: true` | Tool blocked |
| 3 | `continue: true` | `deny` | Tool blocked |
| 4 | `deny` | `deny` | Tool blocked |

In [ ]:
# Setup — imports, plugin paths, helpers
import json
from pathlib import Path
from lib import run_claude

PLUGIN_A = Path("plugins/hook-test-a").resolve()
PLUGIN_B = Path("plugins/hook-test-b").resolve()
LOG_A = Path("/tmp/claude/hook-a-log.jsonl")
LOG_B = Path("/tmp/claude/hook-b-log.jsonl")
RESPONSE_A = Path("/tmp/claude/hook-a-response.json")
RESPONSE_B = Path("/tmp/claude/hook-b-response.json")

CONTINUE = {"continue": True}
DENY = {
    "hookSpecificOutput": {
        "hookEventName": "PreToolUse",
        "permissionDecision": "deny",
        "permissionDecisionReason": "Blocked by test hook."
    }
}

TOOL_PROMPT = "Use Bash to run: echo hello"

def clear_logs():
    """Remove prior log and response files."""
    for f in [LOG_A, LOG_B, RESPONSE_A, RESPONSE_B]:
        f.unlink(missing_ok=True)

def set_responses(a_response: dict, b_response: dict):
    """Write response configs for both hooks."""
    RESPONSE_A.parent.mkdir(parents=True, exist_ok=True)
    RESPONSE_A.write_text(json.dumps(a_response))
    RESPONSE_B.write_text(json.dumps(b_response))

def load_log(path: Path) -> list[dict]:
    """Load log entries from a JSONL file."""
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line]

def run_test(a_response: dict, b_response: dict) -> dict:
    """Configure hooks, run claude -p with both plugins, return results."""
    clear_logs()
    set_responses(a_response, b_response)
    
    # Note: claude -p only accepts one --plugin-dir, so we test with
    # a wrapper approach. If --plugin-dir doesn't support multiple dirs,
    # we may need an alternative strategy.
    import subprocess, uuid
    session_id = str(uuid.uuid4())
    cmd = [
        "claude", "-p", TOOL_PROMPT,
        "--session-id", session_id,
        "--output-format", "json",
        "--plugin-dir", str(PLUGIN_A),
        "--plugin-dir", str(PLUGIN_B),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    output = json.loads(result.stdout) if result.stdout.strip() else {"error": result.stderr}
    
    log_a = load_log(LOG_A)
    log_b = load_log(LOG_B)
    
    return {
        "output": output,
        "returncode": result.returncode,
        "log_a": log_a,
        "log_b": log_b,
    }

print(f"Plugin A: {PLUGIN_A}")
print(f"Plugin B: {PLUGIN_B}")
print("Ready.")

### Pre-check: Does `--plugin-dir` accept multiple values?

Before running the interaction tests, we need to confirm that `claude -p` can load two plugins simultaneously. If `--plugin-dir` only accepts one value, we'll need an alternative approach (e.g., a wrapper plugin that loads both).

In [ ]:
# Pre-check: test if --plugin-dir can be specified twice
import subprocess

check = subprocess.run(
    ["claude", "--help"],
    capture_output=True, text=True
)
help_text = check.stdout + check.stderr

# Look for plugin-dir documentation
for line in help_text.splitlines():
    if "plugin" in line.lower():
        print(line.strip())

# If --plugin-dir doesn't support multiple values, we need a fallback.
# Fallback: create a combined plugin that loads both hook sets.
print("\nIf multiple --plugin-dir is unsupported, see fallback cell below.")

In [ ]:
# Fallback: create a combined plugin that registers both hooks
# Only needed if --plugin-dir doesn't accept multiple values
import shutil

COMBINED_DIR = Path("/tmp/claude/combined-hook-test")

def create_combined_plugin():
    """Create a single plugin that registers both hook A and hook B."""
    if COMBINED_DIR.exists():
        shutil.rmtree(COMBINED_DIR)
    
    # Plugin manifest
    plugin_dir = COMBINED_DIR / ".claude-plugin"
    plugin_dir.mkdir(parents=True)
    (plugin_dir / "plugin.json").write_text(json.dumps({
        "name": "combined-hook-test",
        "version": "0.1.0",
        "description": "Combined plugin with two PreToolUse hooks for interaction testing"
    }))
    
    # Hooks config — two hooks in the same PreToolUse array
    hooks_dir = COMBINED_DIR / "hooks"
    hooks_dir.mkdir()
    (hooks_dir / "hooks.json").write_text(json.dumps({
        "description": "Two PreToolUse hooks to test interaction behavior",
        "hooks": {
            "PreToolUse": [
                {
                    "hooks": [
                        {
                            "type": "command",
                            "command": str(PLUGIN_A / "scripts" / "hook-a.py")
                        }
                    ]
                },
                {
                    "hooks": [
                        {
                            "type": "command",
                            "command": str(PLUGIN_B / "scripts" / "hook-b.py")
                        }
                    ]
                }
            ]
        }
    }, indent=2))
    
    print(f"Combined plugin created at: {COMBINED_DIR}")
    return COMBINED_DIR

combined = create_combined_plugin()
print("Fallback combined plugin ready.")

In [ ]:
# Updated run_test using the combined plugin
from lib import run_claude

def run_interaction_test(a_response: dict, b_response: dict, label: str = "") -> dict:
    """Configure hooks, run claude -p with combined plugin, return results."""
    clear_logs()
    set_responses(a_response, b_response)
    
    cr = run_claude(TOOL_PROMPT, plugin_dir=COMBINED_DIR)
    
    log_a = load_log(LOG_A)
    log_b = load_log(LOG_B)
    
    result = {
        "label": label,
        "returncode": cr.returncode,
        "log_a_count": len(log_a),
        "log_b_count": len(log_b),
        "log_a": log_a,
        "log_b": log_b,
    }
    
    # Check if Bash actually ran by looking at agent output
    raw = json.dumps(cr.output)
    result["bash_ran"] = "hello" in raw.lower()
    result["denied"] = "denied" in raw.lower() or "blocked" in raw.lower()
    
    return result

print("run_interaction_test() ready.")

## Test 1: Both hooks allow

In [ ]:
# Test 1: Both hooks return continue: true
print("="*60)
print("TEST 1: Both hooks ALLOW")
print("  Hook A: continue: true")
print("  Hook B: continue: true")
print("="*60)

r1 = run_interaction_test(CONTINUE, CONTINUE, "both-allow")

print(f"\nBash ran: {r1['bash_ran']}")
print(f"Denied: {r1['denied']}")
print(f"Hook A fired: {r1['log_a_count']} time(s)")
print(f"Hook B fired: {r1['log_b_count']} time(s)")

# Check ordering
if r1['log_a'] and r1['log_b']:
    a_times = [e['timestamp'] for e in r1['log_a']]
    b_times = [e['timestamp'] for e in r1['log_b']]
    print(f"\nHook A timestamps: {a_times}")
    print(f"Hook B timestamps: {b_times}")
    if a_times[0] < b_times[0]:
        print("Ordering: A fires BEFORE B")
    else:
        print("Ordering: B fires BEFORE A")

## Test 2: First hook denies

In [ ]:
# Test 2: Hook A denies, Hook B allows
DENY_A = {
    "hookSpecificOutput": {
        "hookEventName": "PreToolUse",
        "permissionDecision": "deny",
        "permissionDecisionReason": "Blocked by HOOK A."
    }
}

print("="*60)
print("TEST 2: Hook A DENIES, Hook B allows")
print("="*60)

r2 = run_interaction_test(DENY_A, CONTINUE, "a-denies")

print(f"\nBash ran: {r2['bash_ran']}")
print(f"Denied: {r2['denied']}")
print(f"Hook A fired: {r2['log_a_count']} time(s)")
print(f"Hook B fired: {r2['log_b_count']} time(s)")

# KEY QUESTION: Did Hook B fire at all?
if r2['log_b_count'] == 0:
    print("\n>>> SHORT-CIRCUIT: Hook A denial prevented Hook B from running")
else:
    print("\n>>> NO SHORT-CIRCUIT: Both hooks ran despite Hook A denial")

## Test 3: Second hook denies

In [ ]:
# Test 3: Hook A allows, Hook B denies
DENY_B = {
    "hookSpecificOutput": {
        "hookEventName": "PreToolUse",
        "permissionDecision": "deny",
        "permissionDecisionReason": "Blocked by HOOK B."
    }
}

print("="*60)
print("TEST 3: Hook A allows, Hook B DENIES")
print("="*60)

r3 = run_interaction_test(CONTINUE, DENY_B, "b-denies")

print(f"\nBash ran: {r3['bash_ran']}")
print(f"Denied: {r3['denied']}")
print(f"Hook A fired: {r3['log_a_count']} time(s)")
print(f"Hook B fired: {r3['log_b_count']} time(s)")

# Both should fire since A allowed
if r3['log_a_count'] > 0 and r3['log_b_count'] > 0:
    print("\n>>> Both hooks ran — B's denial still blocked the tool")

## Test 4: Both hooks deny

In [ ]:
# Test 4: Both hooks deny
print("="*60)
print("TEST 4: Both hooks DENY")
print("="*60)

r4 = run_interaction_test(DENY_A, DENY_B, "both-deny")

print(f"\nBash ran: {r4['bash_ran']}")
print(f"Denied: {r4['denied']}")
print(f"Hook A fired: {r4['log_a_count']} time(s)")
print(f"Hook B fired: {r4['log_b_count']} time(s)")

# Which denial reason was surfaced to the agent?
if r4['log_a_count'] > 0 and r4['log_b_count'] == 0:
    print("\n>>> SHORT-CIRCUIT: Only A's denial reason surfaced")
elif r4['log_a_count'] > 0 and r4['log_b_count'] > 0:
    print("\n>>> BOTH RAN: Both denial reasons may be surfaced")

## Analysis: Ordering consistency

In [ ]:
# Analyze ordering across all tests
print("ORDERING ANALYSIS")
print("="*60)

results = [r1, r2, r3, r4]

for r in results:
    label = r['label']
    a_fired = r['log_a_count'] > 0
    b_fired = r['log_b_count'] > 0
    
    if a_fired and b_fired:
        a_first = r['log_a'][0]['timestamp']
        b_first = r['log_b'][0]['timestamp']
        order = "A→B" if a_first < b_first else "B→A"
        delta_ms = abs(b_first - a_first) * 1000
        print(f"  {label}: {order} (Δ={delta_ms:.1f}ms)")
    elif a_fired and not b_fired:
        print(f"  {label}: A only (B short-circuited)")
    elif b_fired and not a_fired:
        print(f"  {label}: B only (A short-circuited)")
    else:
        print(f"  {label}: Neither fired")

print("\nSUMMARY")
print("-"*60)
print(f"Tool blocked when A denies: {r2['denied']}")
print(f"Tool blocked when B denies: {r3['denied']}")
print(f"Short-circuit on first denial: {r2['log_b_count'] == 0}")

## Findings

Fill in after running the experiments above.

### Q5: How do multiple PreToolUse hooks on the same tool interact?

| Behavior | Observed |
|----------|----------|
| Ordering | |
| Ordering deterministic? | |
| Short-circuit on denial? | |
| Both denials surfaced? | |
| Hooks see each other's responses? | |

### Implications for rodin

- If short-circuiting occurs, rodin's hook ordering relative to other ExitPlanMode hooks matters
- If both run regardless, rodin can coexist with other hooks without ordering concerns
- If hooks see each other's responses, there may be coordination opportunities